In [ ]:
#RAG

45

In [2]:
import os
from getpass import getpass

from dotenv import load_dotenv
load_dotenv()

MESHAPI_BASE_URL = os.getenv("MESHAPI_BASE_URL", "https://api.meshapi.ai")
MESHAPI_TOKEN = os.getenv("MESH_API_KEY") or os.getenv("MESHAPI_TOKEN") or getpass("MeshAPI token (rsk_...): ")
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY") or getpass("Pinecone API key: ")


In [5]:
PINECONE_INDEX_NAME = "meshapi-demo-kb"
PINECONE_CLOUD = "aws"
PINECONE_REGION = "us-east-1"
EMBEDDING_DIMENSIONS = 1024

In [6]:
# OPEN THE MESHAPI CLIENT

from meshapi import MeshAPI

# Native MeshAPI SDK client -- used for chat, embeddings, and model discovery
client = MeshAPI(base_url=MESHAPI_BASE_URL, token=MESHAPI_TOKEN)
print("MeshAPI client ready.")


MeshAPI client ready.


In [ ]:
#MODELS



In [ ]:
FAST_MODEL = "openai/gpt-4o-mini"
SMART_MODEL = "mistral/mistral-large-3-675b-instruct"
EMBEDDING_MODEL = "openai/text-embedding-3-small"


In [7]:
# ask() helper -- one chat completion, any model

from meshapi import ChatCompletionParams, ChatMessage

def ask(model, prompt, temperature=0.4, max_tokens=350):
    resp = client.chat.completions.create(
        ChatCompletionParams(
            model=model,
            messages=[ChatMessage(role="user", content=prompt)],
            temperature=temperature,
            max_tokens=max_tokens,
        )
    )
    return resp.choices[0].message.content


In [ ]:
#EMBEDDINGS


In [8]:
from meshapi import EmbeddingsParams

def mesh_embed(texts):
    resp = client.embeddings.create(
        EmbeddingsParams(model=EMBEDDING_MODEL, input=texts, dimensions=EMBEDDING_DIMENSIONS)
    )
    return [d.embedding for d in sorted(resp.data, key=lambda d: d.index)]


In [11]:
import time

from pinecone import Pinecone, ServerlessSpec

pc = Pinecone(api_key=PINECONE_API_KEY)

if PINECONE_INDEX_NAME not in [idx["name"] for idx in pc.list_indexes()]:
    pc.create_index(
        name=PINECONE_INDEX_NAME,
        dimension=EMBEDDING_DIMENSIONS,
        metric="cosine",
        spec=ServerlessSpec(cloud=PINECONE_CLOUD, region=PINECONE_REGION),
    )
    while not pc.describe_index(PINECONE_INDEX_NAME).status["ready"]:
        time.sleep(1)

index = pc.Index(PINECONE_INDEX_NAME)
print(index.describe_index_stats())


DescribeIndexStatsResponse(dimension=1024, total_vector_count=0, metric='cosine', namespaces=0)
